# Qubit-number upper bound

This notebook implements the corrected appendix formula with $L=2^{fr}$ and $k=2n-r$. The lengths are assembled from the same helpers used by the gate estimators, including $l_{L_j}=\lfloor\log_2L_j\rfloor+l_p+1$ and $l_{u_j}=l_M-l_{L_j}$. The small term $\log_2(1-2^{-1/(kL)})$ is evaluated stably without constructing $L$.

In [1]:
import math

import Q_Toffoli_cost as QTC


def _select_f(n, case=None):
    """Return f; case=1/2 is retained for backward compatibility."""
    if case == 1:
        return 1.302
    if case == 2:
        return 1.108
    if case is not None:
        raise ValueError("case must be 1, 2, or None")
    return QTC.kyber_instance_parameters(n)[0]


def total_qubit_number(n, r, case=None):
    """Compute the appendix upper bound for the total qubit count."""
    if not 1 <= r <= n:
        raise ValueError("require 1 <= r <= n")

    f = _select_f(n, case)
    k = QTC.compute_k(n, r)
    l_r = QTC.compute_l_r(r)
    l_k = QTC.compute_l_k(k)
    l_t = QTC.compute_l_t(l_r)
    l_tr = QTC.compute_l_tr(l_r)
    l_p = QTC.compute_l_p(l_t, f, r, k)
    l_tilde_b = QTC.compute_l_tilde_b(l_p)
    l_t_tilde_b = QTC.compute_l_t_tilde_b(l_t, l_tilde_b)
    l_M = QTC.compute_l_M(l_t_tilde_b, l_k)

    # The data-independent appendix bound sets L_j=1 for every j.
    max_l_u = l_M - QTC.compute_l_L_j(l_p, 0.0)
    direct = (
        2 * k * l_t_tilde_b
        + k * l_tr
        + k * l_t
        + k * l_M
        + k
        + 2 * max_l_u
        + 3 * r
        + 1
    )

    # Independent check against the boxed closed form in the appendix.
    lam = l_p - (l_t - 1)
    closed = (
        (316 + 16 * l_r + 6 * lam + 2 * l_k) * n
        - (155 + 8 * l_r + 3 * lam + l_k) * r
        + 51 + 2 * l_r + 2 * l_k
    )
    if direct != closed:
        raise AssertionError("direct and closed-form qubit bounds disagree")
    return direct


def find_max_qn(n, case=None):
    values = ((total_qubit_number(n, r, case), r) for r in range(1, n))
    return max(values)


def compute_max_qn(n, case=None):
    max_qn, max_r = find_max_qn(n, case)
    print(
        f"Max qubit number for n={n}: {max_qn} "
        f"(log2 scale: {math.log2(max_qn):.6f}) at r={max_r}"
    )
    return max_qn, max_r


for n in (256, 512, 768, 1024):
    compute_max_qn(n)


Max qubit number for n=256: 334762 (log2 scale: 18.352776) at r=219
Max qubit number for n=512: 1184159 (log2 scale: 20.175431) at r=478
Max qubit number for n=768: 2208481 (log2 scale: 21.074623) at r=714
Max qubit number for n=1024: 3817197 (log2 scale: 21.864082) at r=976
